# Transcription Adapter

> The typed transcription task contract — `TranscriptionAdapter` ABC +
> the provisional tool protocol (capability-unit Option C, pass-2 Thread 3).

In [ ]:
#| default_exp adapter

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import abstractmethod
from pathlib import Path
from typing import Any, ClassVar, Dict, Protocol, Union, runtime_checkable

from cjm_substrate.core.adapter import TaskAdapter

# Canonical home (stage 8 / PILLAR 1c): import the data noun directly, not via
# the REMOVE-AFTER-OVERHAUL `.core` shim, so the permanent ABC survives the
# shim's retirement.
from cjm_capability_primitives.transcription import TranscriptionResult

## Tool protocol (BORN REAL — stage 8)

`required_tool_protocol` is now the REAL structural contract, derived from the native tool surface the Option C migration establishes: a transcription tool capability exposes a pure-compute `transcribe(audio, **kwargs) -> TranscriptionResult` plus `get_current_config()` (the adapter hashes the effective config for its cache key). It REPLACES the provisional `execute`-shaped mirror. `db_path` is deliberately NOT on the protocol — the adapter resolves storage from the substrate-injected `PLUGIN_DATA_DIR` (the substrate owns the data-dir convention), so persistence stays off the pure-compute tool.

Surface-based compatibility (stage 4) gates auto-binding: only tools whose recorded structural surface carries `transcribe` + `get_current_config` match — so the generic adapter binds to MIGRATED tools and skips fused-era `execute` tools (correct: their cache still lives on the tool until they migrate).

In [ ]:
#| export
@runtime_checkable
class TranscriptionToolProtocol(Protocol):
    """Structural contract for transcription tool capabilities (BORN REAL at
    stage 8 — derived from the native tool surface, replacing the provisional
    `execute`-shaped mirror).

    Pure compute: `transcribe` loads the model + runs inference + builds the
    typed result. `get_current_config` supplies the effective config the
    generic adapter hashes for its cache key. Persistence is NOT here — the
    adapter owns it (the native-surface seam)."""
    def transcribe(self, audio: Union[str, Path], **kwargs) -> TranscriptionResult: ...
    def get_current_config(self) -> Dict[str, Any]: ...

In [ ]:
#| export
class TranscriptionAdapter(TaskAdapter):
    """Typed transcription task adapter: model-ready audio in,
    `TranscriptionResult` out.

    Input contract (carried over from the fused-era TranscriptionPlugin):
    the caller guarantees MODEL-READY audio — format / sample-rate /
    channel handling happens upstream (ffmpeg `convert_for_model`), never
    in the adapter.

    Native-surface model (stage 8 / PILLAR 1c): the TOOL is pure compute;
    the ADAPTER owns the cache + persistence bookends (see
    `GenericTranscriptionAdapter`) + the per-call `force` control. Storage
    resolves from the substrate-injected `PLUGIN_DATA_DIR`; `db_path` is not
    on the tool protocol.

    Implementations run in-worker beside their tool capability and are
    constructed with the bound tool instance: `AdapterClass(tool)` (mirrors
    `GraphStorageAdapter`). The result DTO is wire-registered
    ("transcription.result"): returned values cross the worker boundary typed.
    """

    task_name: ClassVar[str] = "transcription"
    required_tool_protocol: ClassVar[type] = TranscriptionToolProtocol

    def __init__(
        self,
        tool: TranscriptionToolProtocol,  # The bound tool capability instance (worker-side binding)
    ):
        self.tool = tool

    @abstractmethod
    def transcribe(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        **kwargs,                 # Provenance (job_id / source_*_time) + tool options
    ) -> TranscriptionResult:     # Typed transcription output
        """Transcribe model-ready audio to text."""
        ...

In [ ]:
# Shape tests: abstract gate + ClassVars + __init__(tool) + a concrete impl
# returning the typed DTO + the REAL protocol structurally matching a migrated
# tool surface (transcribe + get_current_config), and NOT a fused-era execute tool.
class _FakeTool:  # structural match, no inheritance
    def transcribe(self, audio, **kwargs) -> TranscriptionResult:
        return TranscriptionResult(text=f"t:{audio}")
    def get_current_config(self): return {"model": "fake"}


class _FakeTranscriptionAdapter(TranscriptionAdapter):
    def transcribe(self, audio, **kwargs) -> TranscriptionResult:
        return self.tool.transcribe(audio, **kwargs)


try:
    TranscriptionAdapter(_FakeTool())  # type: ignore[abstract]
    raise AssertionError("abstract ABC must not instantiate")
except TypeError:
    pass

impl = _FakeTranscriptionAdapter(_FakeTool())
out = impl.transcribe("a.wav")
assert isinstance(out, TranscriptionResult) and out.text == "t:a.wav"
assert TranscriptionAdapter.task_name == "transcription"
assert TranscriptionAdapter.required_tool_protocol is TranscriptionToolProtocol
assert isinstance(_FakeTool(), TranscriptionToolProtocol)

# the provisional execute-shaped tool no longer satisfies the REAL protocol
class _FusedEraTool:
    def execute(self, audio, **kwargs): return {"text": "hi"}
assert not isinstance(_FusedEraTool(), TranscriptionToolProtocol)
print("TranscriptionAdapter shape tests OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()